# 05 - Graph Analytics
Graph A: spasial-temporal (rantai aftershock antar zona). Graph B: bipartite wilayah-karakteristik.

In [ ]:
import os
import sys
import pandas as pd
import networkx as nx

BASE_DIR = os.path.abspath(os.environ.get("BDA_BASE_DIR", ".."))
print("BASE_DIR aktif:", BASE_DIR)
assert os.path.isdir(os.path.join(BASE_DIR, "src")), (
    f"BASE_DIR salah: {BASE_DIR} tidak punya folder src/. "
    "Set os.environ['BDA_BASE_DIR'] ke path repo yang benar sebelum run cell ini."
)

PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
FIG_DIR = os.path.join(BASE_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

sys.path.insert(0, os.path.join(BASE_DIR, "src"))
import graph_analytics as ga

df = pd.read_parquet(os.path.join(PROCESSED_DIR, "earthquake_features.parquet"))
print(f"Loaded: {df.shape}, zona unik: {df['zone_id'].nunique()}")


## Graph A: Spasial-Temporal (zona seismik)

In [ ]:
import time

# node = zona seismik (grid 1 derajat), edge = pasangan zona dengan event berdekatan
# (<=200km & <=7 hari) -- proxy migrasi/rantai aktivitas seismik antar wilayah
t0 = time.time()
G_a = ga.build_graph_a(df, max_distance_km=200, max_days=7)
print(f"Graph A dibangun {time.time()-t0:.1f}s: {G_a.number_of_nodes()} node, {G_a.number_of_edges()} edge")

components = list(nx.connected_components(G_a))
print(f"Jumlah komponen terhubung: {len(components)} (komponen terbesar: {len(max(components, key=len))} zona)")


In [ ]:
metrics_a = ga.compute_graph_a_metrics(G_a)
metrics_a.to_csv(os.path.join(PROCESSED_DIR, "graph_a_metrics.csv"), index=False)

print("Top 10 zona berdasarkan PageRank (paling sentral dalam jaringan aktivitas seismik):")
metrics_a.head(10)


In [ ]:
partition_a = ga.detect_communities(G_a)
n_communities = len(set(partition_a.values()))
nx.set_node_attributes(G_a, partition_a, "community")
print(f"Louvain community detection: {n_communities} komunitas terdeteksi")

community_sizes = pd.Series(partition_a).value_counts().sort_values(ascending=False)
print("Ukuran 10 komunitas terbesar:")
community_sizes.head(10)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

pos = {n: (d["lon"], d["lat"]) for n, d in G_a.nodes(data=True)}
node_size = [50 + 20 * G_a.degree(n) for n in G_a.nodes()]
node_color = [partition_a[n] for n in G_a.nodes()]

fig, ax = plt.subplots(figsize=(10, 10))
nx.draw_networkx_edges(G_a, pos, alpha=0.2, ax=ax)
nodes = nx.draw_networkx_nodes(
    G_a, pos, node_size=node_size, node_color=node_color, cmap=cm.tab20, ax=ax
)
ax.set_title("Graph A: Jaringan Spasial-Temporal Zona Seismik\n(warna = komunitas Louvain, ukuran = degree)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "graph_a_spatial_temporal.png"), dpi=150)
plt.show()


## Graph B: Bipartite Wilayah-Karakteristik

In [ ]:
# node = wilayah (zone_id) + kategori (depth_class, mag_band, tsunami-flag)
# edge = jumlah kejadian yang menghubungkan wilayah ke kategori tsb
G_b = ga.build_graph_b(df)
print(f"Graph B dibangun: {G_b.number_of_nodes()} node, {G_b.number_of_edges()} edge")

wilayah_nodes = [n for n, d in G_b.nodes(data=True) if d.get("node_type") == "wilayah"]
kategori_nodes = [n for n, d in G_b.nodes(data=True) if d.get("node_type") == "kategori"]
print(f"Node wilayah: {len(wilayah_nodes)}, node kategori: {len(kategori_nodes)}")
print("Kategori:", kategori_nodes)


In [ ]:
# Wilayah multi-hazard: wilayah yang terhubung ke banyak kategori berbeda
# (mengalami kombinasi depth/magnitude/tsunami yang beragam)
mh_degree = ga.compute_zone_multihazard_degree(G_b)
mh_degree.to_csv(os.path.join(PROCESSED_DIR, "graph_b_multihazard.csv"), index=False)
print("Top 10 wilayah paling beragam karakteristik hazard-nya:")
mh_degree.head(10)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 10))
pos_b = nx.spring_layout(G_b, k=0.3, seed=42)

nx.draw_networkx_nodes(
    G_b, pos_b, nodelist=wilayah_nodes, node_color="steelblue",
    node_size=40, alpha=0.6, label="Wilayah", ax=ax,
)
nx.draw_networkx_nodes(
    G_b, pos_b, nodelist=kategori_nodes, node_color="orangered",
    node_size=300, alpha=0.9, label="Kategori", ax=ax,
)
nx.draw_networkx_labels(
    G_b, pos_b, labels={n: n for n in kategori_nodes}, font_size=8, ax=ax,
)
nx.draw_networkx_edges(G_b, pos_b, alpha=0.05, ax=ax)

ax.set_title("Graph B: Bipartite Wilayah - Karakteristik Hazard")
ax.legend()
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "graph_b_bipartite.png"), dpi=150)
plt.show()


In [ ]:
# Insight naratif ringkas -- angka aktual, diisi otomatis dari hasil run
print("=== INSIGHT GRAPH A (spasial-temporal) ===")
print(f"- Jaringan terpecah jadi {len(components)} komponen terhubung dari {G_a.number_of_nodes()} zona.")
print(f"  Komponen terbesar hanya {len(max(components, key=len))} zona -> aktivitas seismik Indonesia")
print(f"  didominasi klaster LOKAL per wilayah, bukan satu jaringan besar yang saling terhubung.")
print(f"- Zona dengan PageRank tertinggi (paling sentral):")
for _, row in metrics_a.head(3).iterrows():
    print(f"    {row['zone_id']}: pagerank={row['pagerank']:.4f}, degree={row['degree']}")
print(f"- {n_communities} komunitas Louvain terdeteksi -- merepresentasikan kelompok wilayah")
print(f"  dengan pola migrasi aktivitas seismik yang saling terkait secara spasial-temporal.")

print("\n=== INSIGHT GRAPH B (bipartite wilayah-karakteristik) ===")
print(f"Wilayah paling beragam karakteristik hazard-nya (multi-hazard degree tertinggi):")
for _, row in mh_degree.head(3).iterrows():
    print(f"    {row['zone_id']}: terhubung ke {row['multihazard_degree']} kategori berbeda")
print(f"Wilayah ini perlu prioritas mitigasi lebih tinggi karena mengalami kombinasi")
print(f"karakteristik hazard (kedalaman, magnitude, potensi tsunami) yang lebih beragam")
print(f"dibanding wilayah lain -- bukan cuma frekuensi kejadian tinggi.")


In [ ]:
# Push hasil (csv metrik, figures) ke GitHub supaya bisa dicek tanpa Drive
import sync
sync.push_results(BASE_DIR, message="Update hasil 05_graph_analytics")
